In [2]:
import os
import pandas as pd
import geopandas as gpd
# Allow more columns to be displayed
pd.set_option("display.max_columns", None)

import logging
logging.basicConfig(level=logging.WARNING)

%load_ext autoreload
%autoreload 2

In [3]:
!which python

/Users/klemenkubelj/miniconda3/envs/cvar-masters/bin/python


In [4]:
DATA_DIR = "/Users/klemenkubelj/Documents/school/graduate/masters/data/AJDA_izvoz_podatkov"

In [5]:
vloge_dfs = []
postavka3_dfs = []
postavka4_dfs = []

events = (
    "0014",
    "0018",
    "0024",
    "0068",
)
for event in events:
    path = os.path.join(DATA_DIR, f"Izvoz_podatkov_{event}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"File {path} does not exist")
    
    _vloga_df = pd.read_csv(os.path.join(path, "Vloga.csv"))
    _vloga_df["DogodekId"] = event
    _postavka3_df = pd.read_csv(os.path.join(path, "Postavka3.csv"))
    _postavka3_df["DogodekId"] = event
    _postavka4_df = pd.read_csv(os.path.join(path, "Postavka4.csv"))
    _postavka4_df["DogodekId"] = event
    
    vloge_dfs.append(_vloga_df)
    postavka3_dfs.append(_postavka3_df)
    postavka4_dfs.append(_postavka4_df)

vloge_df = pd.concat(vloge_dfs)
postavka3_df = pd.concat(postavka3_dfs)
postavka4_df = pd.concat(postavka4_dfs)

for event in events:
    print(event)
    print(vloge_df[vloge_df["DogodekId"]==event].shape)
    print(postavka3_df[postavka3_df["DogodekId"]==event].shape)
    print(postavka4_df[postavka4_df["DogodekId"]==event].shape)
    print("-----")


0014
(3145, 74)
(16, 44)
(23226, 34)
-----
0018
(2436, 74)
(36, 44)
(21285, 34)
-----
0024
(2261, 74)
(13, 44)
(12738, 34)
-----
0068
(13084, 74)
(6552, 44)
(68637, 34)
-----


In [6]:
cols_to_keep = [
    "VlogaId",
    # "ZapStev",
    "Objekt_Naslov",
    "Objekt_Naslov_PostnaStevilka",
    "Objekt_Parcela_Stevilka",
    "Objekt_Parcela_KoId",
    "Objekt_UporabnaPovrsina",
    # "Objekt_Id",
    "Objekt_VisinaVodeCm",
    "Objekt_StopnjaPoskodovanosti",
    "Objekt_CentroidX",
    "Objekt_CentroidY",
    "Skoda_DatumOcene",
    "Objekt_SkodaPovzrocenaVPlazu",
    "DogodekId",
]
vloge_df = vloge_df[cols_to_keep]
vloge_df.head(2)

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Objekt_CentroidX,Objekt_CentroidY,Skoda_DatumOcene,Objekt_SkodaPovzrocenaVPlazu,DogodekId
0,147640,Sermin 14,6000 KOPER - CAPODISTRIA,5840/4,715,185.00,30.0,NaN,403438.0,46264.0,09/30/10 00:00:00,NaN,0014
1,147646,Podgorje 10A,6216 PODGORJE,1048/5,694,168.25,10.0,NaN,418447.0,43436.0,09/30/10 00:00:00,NaN,0014


In [7]:
cols_to_keep = [
    "VlogaId",
    "V0",
    "Skoda",
    "OdstPoskodovanostiObjekta",
]
postavka3_df = postavka3_df[cols_to_keep]
postavka3_df.rename(columns={"V0": "Vrednost"}, inplace=True)
postavka3_df.rename(columns={"Skoda": "SkodaPostavka3"}, inplace=True)
postavka3_df.head(2)

,VlogaId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta
0,149162,3309.35,3309.35,100
1,155809,72396.27,72396.27,100


In [8]:
cols_to_keep = [
    "VlogaId",
    "Skoda",
]
postavka4_df = postavka4_df[cols_to_keep]
postavka4_df.head()

# Group by VlogaId and sum Skoda
postavka4_df_sum = postavka4_df.groupby("VlogaId")["Skoda"].sum().reset_index()

# Rename Skoda to SkodaPostavka4Sum
postavka4_df_sum.rename(columns={"Skoda": "SkodaPostavka4Sum"}, inplace=True)
postavka4_df_sum.head(2)

,VlogaId,SkodaPostavka4Sum
0,145913,4359.84
1,145915,289.95


In [9]:
# Merge 3 and 4
postavka34_df = postavka3_df.merge(postavka4_df_sum, on="VlogaId", how="left")
postavka34_df.head(2)

,VlogaId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta,SkodaPostavka4Sum
0,149162,3309.35,3309.35,100,NaN
1,155809,72396.27,72396.27,100,NaN


In [10]:
# Merge vloge_df with postavka34_df
vloge_df = vloge_df.merge(postavka34_df, on="VlogaId", how="left")
vloge_df.head(2)


,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Objekt_CentroidX,Objekt_CentroidY,Skoda_DatumOcene,Objekt_SkodaPovzrocenaVPlazu,DogodekId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta,SkodaPostavka4Sum
0,147640,Sermin 14,6000 KOPER - CAPODISTRIA,5840/4,715,185.00,30.0,NaN,403438.0,46264.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN
1,147646,Podgorje 10A,6216 PODGORJE,1048/5,694,168.25,10.0,NaN,418447.0,43436.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN


In [11]:
# Set new column "SkupnaSkoda". It takes "SkodaPostavka3" if it is not nan, otherwise it takes "SkodaPostavka4Sum"
# If "SkodaPostavka3" was used, set column "SkupnaSkodaSource" to "Postavka3". If "SkodaPostavka4Sum" was used, set it to "Postavka4". If both are nan, set it to nan.

import numpy as np
import pandas as pd

def set_skupna_skoda_and_source(row):
    if pd.notna(row["SkodaPostavka3"]):
        return row["SkodaPostavka3"], "Postavka3"
    elif pd.notna(row["SkodaPostavka4Sum"]):
        return row["SkodaPostavka4Sum"], "Postavka4"
    else:
        return np.nan, np.nan

vloge_df["SkupnaSkoda"], vloge_df["SkupnaSkodaSource"] = zip(*vloge_df.apply(set_skupna_skoda_and_source, axis=1))
vloge_df.head(2)


,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Objekt_CentroidX,Objekt_CentroidY,Skoda_DatumOcene,Objekt_SkodaPovzrocenaVPlazu,DogodekId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta,SkodaPostavka4Sum,SkupnaSkoda,SkupnaSkodaSource
0,147640,Sermin 14,6000 KOPER - CAPODISTRIA,5840/4,715,185.00,30.0,NaN,403438.0,46264.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
1,147646,Podgorje 10A,6216 PODGORJE,1048/5,694,168.25,10.0,NaN,418447.0,43436.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Check how many rows have "SkupnaSkoda" not nan
print(vloge_df[vloge_df["SkupnaSkoda"].notna()].shape)
print(vloge_df[vloge_df["SkupnaSkoda"].isna()].shape)


(6615, 19)
(14311, 19)


### Preprocessing

In [14]:
vloge_df.head()

,VlogaId,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Objekt_CentroidX,Objekt_CentroidY,Skoda_DatumOcene,Objekt_SkodaPovzrocenaVPlazu,DogodekId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta,SkodaPostavka4Sum,SkupnaSkoda,SkupnaSkodaSource
0,147640,Sermin 14,6000 KOPER - CAPODISTRIA,5840/4,715,185.00,30.0,NaN,403438.0,46264.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
1,147646,Podgorje 10A,6216 PODGORJE,1048/5,694,168.25,10.0,NaN,418447.0,43436.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
2,147651,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.00,30.0,NaN,405601.0,46361.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
3,147686,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.00,50.0,NaN,405580.0,46339.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
4,147692,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.00,50.0,NaN,405580.0,46339.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# 1. Keep rows where `Objekt_SkodaPovzrocenaVPlazu` is 0 or na
vloge_df = vloge_df[
    (vloge_df["Objekt_SkodaPovzrocenaVPlazu"]==0) | (vloge_df["Objekt_SkodaPovzrocenaVPlazu"].isna())
]
# 2. Remove where `Objekt_Id` is nan or is duplicated
# vloge_df = vloge_df[
#     vloge_df["Objekt_Id"].notna()
#     & ~vloge_df["Objekt_Id"].duplicated()
# ].shape()
# 3. Cast
# vloge_df["Objekt_Id"] = vloge_df["Objekt_Id"].astype(int)
vloge_df.shape

(20040, 20)

In [135]:
vloge_df[
    vloge_df["Objekt_CentroidX"].notna()
].shape

(7859, 21)

In [134]:
vloge_df.head()

,VlogaId,ZapStev,Objekt_Naslov,Objekt_Naslov_PostnaStevilka,Objekt_Parcela_Stevilka,Objekt_Parcela_KoId,Objekt_UporabnaPovrsina,Objekt_Id,Objekt_VisinaVodeCm,Objekt_StopnjaPoskodovanosti,Objekt_CentroidX,Objekt_CentroidY,Skoda_DatumOcene,Objekt_SkodaPovzrocenaVPlazu,DogodekId,Vrednost,SkodaPostavka3,OdstPoskodovanostiObjekta,SkodaPostavka4Sum,SkupnaSkoda,SkupnaSkodaSource
0,147640,4,Sermin 14,6000 KOPER - CAPODISTRIA,5840/4,715,185.00,10005913.0,30.0,NaN,403438.0,46264.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
1,147646,5,Podgorje 10A,6216 PODGORJE,1048/5,694,168.25,20322111.0,10.0,NaN,418447.0,43436.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
2,147651,6,Cesta med vinogradi 48,6000 KOPER - CAPODISTRIA,5743/6,715,50.00,10005884.0,30.0,NaN,405601.0,46361.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
3,147686,7,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,55.00,10005892.0,50.0,NaN,405580.0,46339.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN
4,147692,8,Cesta med vinogradi 46,6000 KOPER - CAPODISTRIA,5743/10,715,38.00,10005892.0,50.0,NaN,405580.0,46339.0,09/30/10 00:00:00,NaN,0014,NaN,NaN,NaN,NaN,NaN,NaN


### Export

In [145]:
for c in vloge_df.columns:
    print(f'"{c}",')

"VlogaId",
"Objekt_Naslov",
"Objekt_Naslov_PostnaStevilka",
"Objekt_Parcela_Stevilka",
"Objekt_Parcela_KoId",
"Objekt_UporabnaPovrsina",
"Objekt_VisinaVodeCm",
"Objekt_StopnjaPoskodovanosti",
"Objekt_CentroidX",
"Objekt_CentroidY",
"Skoda_DatumOcene",
"Objekt_SkodaPovzrocenaVPlazu",
"DogodekId",
"Vrednost",
"SkodaPostavka3",
"OdstPoskodovanostiObjekta",
"SkodaPostavka4Sum",
"SkupnaSkoda",
"SkupnaSkodaSource",


In [149]:
cols_to_keep = [
    "VlogaId",
    "Objekt_Naslov",
    "Objekt_Naslov_PostnaStevilka",
    "Objekt_Parcela_Stevilka",
    "Objekt_Parcela_KoId",
    "Objekt_UporabnaPovrsina",
    "Objekt_VisinaVodeCm",
    "Objekt_StopnjaPoskodovanosti",
    "Objekt_CentroidX",
    "Objekt_CentroidY",
    "Skoda_DatumOcene",
    "Objekt_SkodaPovzrocenaVPlazu",
    "DogodekId",
    "Vrednost",
    # "SkodaPostavka3",
    "OdstPoskodovanostiObjekta",
    # "SkodaPostavka4Sum",
    "SkupnaSkoda",
    "SkupnaSkodaSource",
]
vloge_df = vloge_df[cols_to_keep]

# Export to csv
vloge_df.to_csv(os.path.join("../data/vloge_processed_2025-05-10.csv"), index=False)
